In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Dataset

# Define Barlow Twins Loss
class BarlowTwinsLoss(nn.Module):
    def __init__(self, lambda_param=0.0051):
        super().__init__()
        self.lambda_param = lambda_param  # Scaling factor for off-diagonal loss

    def forward(self, z1, z2):
        N, D = z1.shape
        z1_norm = (z1 - z1.mean(dim=0)) / z1.std(dim=0)
        z2_norm = (z2 - z2.mean(dim=0)) / z2.std(dim=0)

        # Compute cross-correlation matrix
        c = (z1_norm.T @ z2_norm) / N

        # Identity matrix for decorrelation
        I = torch.eye(D, device=z1.device)

        # Compute loss: diagonal (alignment) + off-diagonal (decorrelation)
        loss = ((1 - c.diagonal()) ** 2).sum() + self.lambda_param * ((c - I) ** 2).sum()
        return loss

# Define Simple Encoder with MLP Projection Head
class BarlowTwins(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.backbone = models.resnet18(pretrained=False)
        self.backbone.fc = nn.Identity()  # Remove classification head

        # Projection Head (MLP)
        self.projector = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, feature_dim)
        )

    def forward(self, x1, x2):
        z1 = self.projector(self.backbone(x1))
        z2 = self.projector(self.backbone(x2))
        return z1, z2

# Augmentation Pipeline
transform = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

# Custom Dataset Wrapper for Two Augmented Views
class AugmentedDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, index):
        img, _ = self.dataset[index]
        return transform(img), transform(img)  # Two different views

    def __len__(self):
        return len(self.dataset)

# Load CIFAR-10 dataset
dataset = AugmentedDataset(CIFAR10(root="./data", train=True, download=True))
train_loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2)

# Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BarlowTwins().to(device)
criterion = BarlowTwinsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
for epoch in range(10):
    total_loss = 0
    for x1, x2 in train_loader:
        x1, x2 = x1.to(device), x2.to(device)

        # Forward pass
        z1, z2 = model(x1, x2)
        loss = criterion(z1, z2)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/10], Loss: {total_loss/len(train_loader):.4f}")

